<a href="https://colab.research.google.com/github/Masiania-bit/Maria-palander/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 2: Data Analysis — Чтение и проверка данных

**Цель**: Научиться читать CSV-файлы из репозитория GitHub в Google Colab и выполнять базовую проверку данных с помощью pandas.

**Данные:**
- `volcano.csv` — информация о вулканах: название, высота, континент, горная цепь

**Что мы делаем:**
1. Клонируем ваш репозиторий GitHub в Colab
2. Читаем файл вулканов в pandas DataFrame
3. Очищаем и анализируем структуру столбцов
4. Проверяем данные: пропуски, типы, статистику по высоте вулканов

## 🐱 [1] Клонируем репозиторий курса в Colab

In [3]:
# 🐱 Шаг 1. Клонируем ваш репозиторий в Colab

import os

if not os.path.exists("Maria-palander"):
    !git clone -q https://github.com/Masiania-bit/Maria-palander.git

%cd Maria-palander

print("✅ Репозиторий готов, теперь мы работаем внутри папки Maria-palander")

/content/Maria-palander
✅ Репозиторий готов, теперь мы работаем внутри папки Maria-palander


## 📥 [2A] Простое чтение CSV-файлов в pandas

Сначала просто прочитаем оба CSV-файла в объекты `DataFrame`, без каких‑либо изменений.

После этого мы узнаем, сколько строк загружено в каждый датасет.

In [4]:
# 🌋 Шаг 2. Чтение данных о вулканах в pandas

import pandas as pd

df_volcano = pd.read_csv("data/volcano.csv")

print("✅ Загружено строк в датасете:", len(df_volcano))
print("\n📋 Структура данных:")
print(df_volcano.info())

print("\n🔍 Первые 5 строк:")
print(df_volcano.head())

✅ Загружено строк в датасете: 2388

📋 Структура данных:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2388 entries, 0 to 2387
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   volcano             2388 non-null   object 
 1   volcanoLabel        2388 non-null   object 
 2   elevation           1362 non-null   float64
 3   country             1927 non-null   object 
 4   countryLabel        1927 non-null   object 
 5   coord               2338 non-null   object 
 6   mountainRange       736 non-null    object 
 7   mountainRangeLabel  736 non-null    object 
dtypes: float64(1), object(7)
memory usage: 149.4+ KB
None

🔍 Первые 5 строк:
                                    volcano     volcanoLabel  elevation  \
0    http://www.wikidata.org/entity/Q499164  Гора Аскрийская    18225.0   
1   http://www.wikidata.org/entity/Q2065108     Mount Tehama     9239.0   
2   http://www.wikidata.org/entity/Q3467801  

## 🧹 [2B] Очистка и переименование столбцов

В исходном CSV-файле с данными о вулканах есть **столбцы из Викиданных** с постфиксом `Label`, которые содержат читаемые названия, но имеют неудобные имена для анализа:

- `volcanoLabel` — название вулкана (читаемое имя)
- `continentLabel` — континент
- `mountainRangeLabel` — горная цепь
- `elevation` — высота вулкана в метрах (числовой столбец)

В этом шаге мы:
- переименуем столбцы, убрав постфикс `Label`:  
  `volcanoLabel → volcano`, `continentLabel → continent`, `mountainRangeLabel → mountainRange`;
- приведём числовой столбец `elevation` к целочисленному типу `int`;
- обработаем пропуски в высоте: некорректные значения преобразуем в `NaN`, затем заменим на 0.

При приведении к числам мы используем:

- `pd.to_numeric(..., errors="coerce")` — преобразует значения в числа, некорректные значения превращает в `NaN`;
- `fillna(0)` — заменяет пропущенные значения (`NaN`) на 0;
- `astype(int)` — переводит столбец к целочисленному типу.

> ⚠️ **Важно:** если в данных есть пропуски в высоте (например, для подводных вулканов), они будут заменены на 0. При углублённом анализе можно рассмотреть другие стратегии заполнения.

In [18]:
# 🧹 Шаг 2B. Очистка столбцов: сохраняем URL, не заменяем пропуски в высоте

# Перечитываем данные
df_volcano = pd.read_csv("data/volcano.csv")

# Переименовываем столбцы: сохраняем URL, убираем постфикс Label
df_volcano = df_volcano.rename(columns={
    "volcano": "URL",              # URI Wikidata вулкана
    "volcanoLabel": "volcano",     # Читаемое название
    "countryLabel": "country",     # Страна
    "mountainRangeLabel": "mountainRange"  # Горная цепь
})

# Удаляем дубликаты столбцов (защита от конфликтов имён)
df_volcano = df_volcano.loc[:, ~df_volcano.columns.duplicated()]

# Приводим высоту к числовому типу, оставляя пропуски как NaN (НЕ заменяем на 0!)
df_volcano["elevation"] = pd.to_numeric(df_volcano["elevation"], errors="coerce")

# Создаём подвыборку только с известной высотой для анализа высотных характеристик
df_with_elev = df_volcano[df_volcano["elevation"].notna()].copy()

# Извлекаем координаты из строки вида "Point(lon lat)" → числовые столбцы lon, lat
df_volcano[["lon", "lat"]] = df_volcano["coord"].str.extract(
    r"Point\(([-\d.]+)\s+([-\d.]+)\)"
).astype(float)

print("✅ Столбцы после очистки:", list(df_volcano.columns))
print(f"\n📊 Всего вулканов: {len(df_volcano)}")
print(f"📊 С известной высотой: {len(df_with_elev)} ({len(df_with_elev)/len(df_volcano)*100:.1f}%)")
print(f"📊 С координатами: {df_volcano['lon'].notna().sum()} ({df_volcano['lon'].notna().mean()*100:.1f}%)")

print("\n🔍 Пример данных после очистки:")
display_cols = ["URL", "volcano", "elevation", "country", "mountainRange", "lon", "lat"]
print(df_volcano[display_cols].head(3).to_string(index=False))

✅ Столбцы после очистки: ['URL', 'volcano', 'elevation', 'country', 'coord', 'mountainRange', 'lon', 'lat']

📊 Всего вулканов: 2388
📊 С известной высотой: 1362 (57.0%)
📊 С координатами: 2338 (97.9%)

🔍 Пример данных после очистки:
                                    URL         volcano  elevation                            country                            mountainRange         lon       lat
 http://www.wikidata.org/entity/Q499164 Гора Аскрийская    18225.0                                NaN                                      NaN  255.920000 11.920000
http://www.wikidata.org/entity/Q2065108    Mount Tehama     9239.0 http://www.wikidata.org/entity/Q30 http://www.wikidata.org/entity/Q27966614 -121.559428 40.445436
http://www.wikidata.org/entity/Q3467801   Hayes Volcano     9147.0 http://www.wikidata.org/entity/Q30   http://www.wikidata.org/entity/Q828541 -152.411000 61.640300


In [19]:
# ⛰️ Шаг 2C. Загрузка данных о горных системах (с проверкой структуры)

df_mr = pd.read_csv("data/mountain_range.csv")

print("Исходные столбцы файла mountain_range.csv:")
print(list(df_mr.columns))
print(f"\n✅ Загружено строк: {len(df_mr)}")

# Проверяем наличие ключевых столбцов перед переименованием
rename_map = {"mountainRange": "URL", "mountainRangeLabel": "mountainRange"}
if "countryLabel" in df_mr.columns:
    rename_map["countryLabel"] = "country"
if "highPointLabel" in df_mr.columns:
    rename_map["highPointLabel"] = "highPoint"
if "partOfLabel" in df_mr.columns:
    rename_map["partOfLabel"] = "partOf"

df_mr = df_mr.rename(columns=rename_map)

# Удаляем дубликаты столбцов
df_mr = df_mr.loc[:, ~df_mr.columns.duplicated()]

# Приводим числовые поля к правильному типу (только если столбец существует)
if "elevation" in df_mr.columns:
    df_mr["elevation"] = pd.to_numeric(df_mr["elevation"], errors="coerce")

# Извлекаем координаты высшей точки (если есть столбец coord)
if "coord" in df_mr.columns:
    coords = df_mr["coord"].str.extract(r"Point\(([-\d.]+)\s+([-\d.]+)\)")
    df_mr["highPoint_lon"] = pd.to_numeric(coords[0], errors="coerce")
    df_mr["highPoint_lat"] = pd.to_numeric(coords[1], errors="coerce")

print("\n📋 Столбцы после очистки:", list(df_mr.columns))

# Выводим первые строки с доступными столбцами
display_cols = ["URL", "mountainRange"]
for col in ["country", "elevation", "highPoint", "partOf", "highPoint_lon", "highPoint_lat"]:
    if col in df_mr.columns:
        display_cols.append(col)

print("\n🔍 Первые 3 строки:")
print(df_mr[display_cols].head(3).to_string(index=False))

# Анализ заполненности доступных полей
print("\n📊 Заполненность ключевых полей в горных системах:")
for col in ["country", "elevation", "partOf", "coord", "highPoint_lon"]:
    if col in df_mr.columns:
        pct = df_mr[col].notna().mean() * 100
        print(f"   • {col:20s}: {pct:5.1f}%")
    else:
        print(f"   • {col:20s}: отсутствует в данных")

Исходные столбцы файла mountain_range.csv:
['volcano', 'volcanoLabel', 'elevation', 'country', 'countryLabel', 'coord', 'mountainRange', 'mountainRangeLabel']

✅ Загружено строк: 2388

📋 Столбцы после очистки: ['volcano', 'volcanoLabel', 'elevation', 'country', 'coord', 'URL', 'mountainRange', 'highPoint_lon', 'highPoint_lat']

🔍 Первые 3 строки:
                                     URL       mountainRange                            country  elevation  highPoint_lon  highPoint_lat
                                     NaN                 NaN                                NaN    18225.0     255.920000      11.920000
http://www.wikidata.org/entity/Q27966614 California Cascades http://www.wikidata.org/entity/Q30     9239.0    -121.559428      40.445436
  http://www.wikidata.org/entity/Q828541 Tordrillo Mountains http://www.wikidata.org/entity/Q30     9147.0    -152.411000      61.640300

📊 Заполненность ключевых полей в горных системах:
   • country             :  80.7%
   • elevation    

## 🔍 [3] Обзор данных: структура и первые строки

Сделаем короткий обзор датафрейма с данными о вулканах:

- посмотрим размер таблицы (`shape`);
- выведем список столбцов после очистки;
- посмотрим первые несколько строк;
- дополнительно посчитаем базовую статистику по высоте вулканов (`elevation`): минимум, максимум, среднее, медиана.

Для удобства напишем функцию `show_info(df, name)`, чтобы структурированно вывести информацию о датафрейме.

In [20]:
# 🔍 Шаг 3. Обзор данных и анализ качества заполнения

def show_info(df, name, n=5):
    """Краткий обзор DataFrame."""
    print(f"\n📊 {name}")
    print("Размер:", df.shape)
    print("Столбцы:", ", ".join(df.columns))
    print("\nПервые строки:")
    print(df.head(n))

# Общий обзор
show_info(df_volcano, "Вулканы (все объекты)")

# Анализ заполненности полей — КЛЮЧЕВОЙ ШАГ ДЛЯ ПЛАНИРОВАНИЯ ВИЗУАЛИЗАЦИЙ
print("\n" + "="*70)
print("📊 АНАЛИЗ ЗАПОЛНЕННОСТИ ПОЛЕЙ (проценты)")
print("="*70)
for col in ["elevation", "country", "mountainRange", "coord", "lon", "lat"]:
    if col in df_volcano.columns:
        pct = df_volcano[col].notna().mean() * 100
        count = df_volcano[col].notna().sum()
        print(f"{col:20s}: {pct:5.1f}% ({count} из {len(df_volcano)})")

# Топ стран и горных систем
print("\n" + "="*70)
print("🌍 ТОП-15 СТРАН ПО ЧИСЛУ ВУЛКАНОВ")
print("="*70)
print(df_volcano["country"].value_counts().head(15).to_string())

print("\n" + "="*70)
print("⛰️  ТОП-15 ГОРНЫХ СИСТЕМ ПО ЧИСЛУ ВУЛКАНОВ")
print("="*70)
print(df_volcano["mountainRange"].value_counts().head(15).to_string())

# Статистика по высоте (только известные значения)
print("\n" + "="*70)
print("📈 СТАТИСТИКА ПО ВЫСОТЕ (только известные значения)")
print("="*70)
print(df_with_elev["elevation"].describe().to_string())


📊 Вулканы (все объекты)
Размер: (2388, 8)
Столбцы: URL, volcano, elevation, country, coord, mountainRange, lon, lat

Первые строки:
                                        URL          volcano  elevation  \
0    http://www.wikidata.org/entity/Q499164  Гора Аскрийская    18225.0   
1   http://www.wikidata.org/entity/Q2065108     Mount Tehama     9239.0   
2   http://www.wikidata.org/entity/Q3467801    Hayes Volcano     9147.0   
3  http://www.wikidata.org/entity/Q12763846  Cerro Nicholson     8282.0   
4    http://www.wikidata.org/entity/Q955807  Isanotski Peaks     8106.0   

                               country  \
0                                  NaN   
1   http://www.wikidata.org/entity/Q30   
2   http://www.wikidata.org/entity/Q30   
3  http://www.wikidata.org/entity/Q419   
4   http://www.wikidata.org/entity/Q30   

                                               coord  \
0  <http://www.wikidata.org/entity/Q111> Point(25...   
1                 Point(-121.559427777 40.445436111

## ✅ [4] Анализ высотных аномалий и распределения вулканов по поясам

В этом шаге мы исследуем два ключевых аспекта высотных данных:

#### 🌌 Экстремальные аномалии (> 8000 м)
Высота Эвереста — 8849 м. Вулканы выше этого порога **не могут существовать на Земле** из-за физических ограничений литосферы. Такие объекты — либо:
- внеземные вулканы (Марс: гора Олимп ~21 км, вулканы Аскрийской равнины),
- ошибки в данных (опечатки, неверные единицы измерения).

Мы выделим все вулканы выше 8000 м и проанализируем их источники.

#### 📏 Высотные пояса вулканической активности
Разобьём вулканы на 5 экологически значимых групп:
- **Подводные** (< 0 м) — вулканы на дне океана;
- **Низкие** (0–1000 м) — прибрежные и равнинные вулканы;
- **Средние** (1000–3000 м) — типичные стратовулканы (Фудзияма — 3776 м);
- **Высокие** (3000–6000 м) — вулканы в горных системах (Килиманджаро — 5895 м);
- **Экстремальные** (> 6000 м) — гиганты, часто на границе ошибок данных.

Этот анализ покажет, как распределена вулканическая активность по вертикали и поможет выявить артефакты в данных.

In [21]:
# ✅ Шаг 4. Анализ высотных аномалий и поясов (только с известной высотой)

print("="*70)
print("🌌 ВУЛКАНЫ-ГИГАНТЫ: выше Эвереста (> 8000 м)")
print("="*70)

extreme_volcanoes = df_with_elev[df_with_elev["elevation"] > 8000].sort_values(
    "elevation", ascending=False
)

if len(extreme_volcanoes) > 0:
    print(f"\n⚠️  Найдено {len(extreme_volcanoes)} вулканов выше 8000 м:\n")
    display_cols = ["volcano", "elevation"]
    if "country" in extreme_volcanoes.columns:
        display_cols.append("country")
    print(extreme_volcanoes[display_cols].to_string(index=False))
else:
    print("\n✅ В датасете нет вулканов выше 8000 м")

# Высотные пояса
def classify_elevation(height):
    if height < 0:
        return "подводные (< 0 м)"
    elif height < 1000:
        return "низкие (0–1000 м)"
    elif height < 3000:
        return "средние (1000–3000 м)"
    elif height < 6000:
        return "высокие (3000–6000 м)"
    else:
        return "экстремальные (> 6000 м)"

df_with_elev["height_belt"] = df_with_elev["elevation"].apply(classify_elevation)

belt_stats = (
    df_with_elev.groupby("height_belt")
    .agg(
        count=("volcano", "count"),
        pct=("volcano", lambda x: f"{len(x) / len(df_with_elev) * 100:.1f}%"),
        min_elev=("elevation", "min"),
        mean_elev=("elevation", "mean"),
        max_elev=("elevation", "max")
    )
    .round({"mean_elev": 1})
    .reindex([
        "подводные (< 0 м)",
        "низкие (0–1000 м)",
        "средние (1000–3000 м)",
        "высокие (3000–6000 м)",
        "экстремальные (> 6000 м)"
    ])
)

print("\n" + "="*70)
print("📏 РАСПРЕДЕЛЕНИЕ ПО ВЫСОТНЫМ ПОЯСАМ")
print("="*70)
print(belt_stats.to_string())

print("\n💡 ВЫВОДЫ ПО ВЫСОТЕ:")
print(f"• Проанализировано {len(df_with_elev)} вулканов с известной высотой")
print(f"• Подводных вулканов: {belt_stats.loc['подводные (< 0 м)', 'count']}")
print(f"• Экстремальных (> 6000 м): {belt_stats.loc['экстремальные (> 6000 м)', 'count']}")

🌌 ВУЛКАНЫ-ГИГАНТЫ: выше Эвереста (> 8000 м)

⚠️  Найдено 5 вулканов выше 8000 м:

        volcano  elevation                             country
Гора Аскрийская    18225.0                                 NaN
   Mount Tehama     9239.0  http://www.wikidata.org/entity/Q30
  Hayes Volcano     9147.0  http://www.wikidata.org/entity/Q30
Cerro Nicholson     8282.0 http://www.wikidata.org/entity/Q419
Isanotski Peaks     8106.0  http://www.wikidata.org/entity/Q30

📏 РАСПРЕДЕЛЕНИЕ ПО ВЫСОТНЫМ ПОЯСАМ
                          count    pct  min_elev  mean_elev  max_elev
height_belt                                                          
подводные (< 0 м)             5   0.4% -1400.000     -722.0      -2.0
низкие (0–1000 м)           453  33.3%     2.014      526.2     999.9
средние (1000–3000 м)       635  46.6%  1000.000     1735.9    2997.0
высокие (3000–6000 м)       214  15.7%  3011.000     4151.7    5988.0
экстремальные (> 6000 м)     55   4.0%  6016.000     6824.9   18225.0

💡 ВЫВОДЫ ПО В

In [22]:
# 🔬 Исследование А: Вулканы выше 8000 м — марсианские объекты

extreme = df_volcano[df_volcano["elevation"] > 8000][["URL", "volcano", "elevation", "country", "lon", "lat"]]
print("Вулканы выше 8000 м:")
print(extreme.to_string(index=False))

print("\n" + "="*70)
print("📝 МОЙ ВЫВОД ПО ИССЛЕДОВАНИЮ А")
print("="*70)
print("""
Я выбираю фильтрацию вулканов выше 7000 м при анализе земных объектов.
Причина: максимальная высота земного вулкана (Охос-дель-Саладо) — 6893 м.
Все объекты выше 7000 м — марсианские вулканы (Аскрийская равнина, Олимп).
Для карты Земли их нужно исключить, но сохранить отдельно как особую группу
для сравнительного анализа планетарной вулканической активности.
""")

Вулканы выше 8000 м:
                                     URL         volcano  elevation                             country         lon        lat
  http://www.wikidata.org/entity/Q499164 Гора Аскрийская    18225.0                                 NaN  255.920000  11.920000
 http://www.wikidata.org/entity/Q2065108    Mount Tehama     9239.0  http://www.wikidata.org/entity/Q30 -121.559428  40.445436
 http://www.wikidata.org/entity/Q3467801   Hayes Volcano     9147.0  http://www.wikidata.org/entity/Q30 -152.411000  61.640300
http://www.wikidata.org/entity/Q12763846 Cerro Nicholson     8282.0 http://www.wikidata.org/entity/Q419  -71.730000 -16.260556
  http://www.wikidata.org/entity/Q955807 Isanotski Peaks     8106.0  http://www.wikidata.org/entity/Q30 -163.729000  54.768600

📝 МОЙ ВЫВОД ПО ИССЛЕДОВАНИЮ А

Я выбираю фильтрацию вулканов выше 7000 м при анализе земных объектов.
Причина: максимальная высота земного вулкана (Охос-дель-Саладо) — 6893 м.
Все объекты выше 7000 м — марсианские ву

In [23]:
# 🔬 Исследование Б: Вулканы без указания страны

no_country = df_volcano[df_volcano["country"].isna()][["URL", "volcano", "elevation", "mountainRange"]].head(15)
print("Примеры вулканов без страны:")
print(no_country.to_string(index=False))

# Анализ типов объектов без страны
submarine_pct = df_volcano[df_volcano["country"].isna() & (df_volcano["elevation"] < 0)].shape[0] / df_volcano[df_volcano["country"].isna()].shape[0] * 100
mars_pct = df_volcano[df_volcano["country"].isna() & (df_volcano["elevation"] > 8000)].shape[0] / df_volcano[df_volcano["country"].isna()].shape[0] * 100

print("\n" + "="*70)
print("📝 МОЙ ВЫВОД ПО ИССЛЕДОВАНИЮ Б")
print("="*70)
print(f"""
Среди {df_volcano['country'].isna().sum()} вулканов без страны:
• ~{submarine_pct:.0f}% — подводные вулканы (высота < 0 м)
• ~{mars_pct:.0f}% — марсианские объекты (высота > 8000 м)
• Остальные — объекты с неполной информацией в Wikidata

Для графиков по странам я буду использовать только вулканы:
1) с известной страной,
2) с высотой 0–7000 м (земные),
3) с координатами для геопривязки.

Это даст ~60% от исходного датасета, но с высоким качеством данных.
""")

Примеры вулканов без страны:
                                      URL          volcano  elevation                          mountainRange
   http://www.wikidata.org/entity/Q499164  Гора Аскрийская    18225.0                                    NaN
  http://www.wikidata.org/entity/Q1472301       Гора Урана     4853.0                                    NaN
   http://www.wikidata.org/entity/Q188982           Эребус     3794.0 http://www.wikidata.org/entity/Q319671
   http://www.wikidata.org/entity/Q380300   Mount Overlord     3395.0                                    NaN
http://www.wikidata.org/entity/Q123682894       Q123682894     3212.0   http://www.wikidata.org/entity/Q5456
 http://www.wikidata.org/entity/Q24044824  Unnamed volcano     2987.0                                    NaN
  http://www.wikidata.org/entity/Q3322058   Mount Rittmann     2600.0 http://www.wikidata.org/entity/Q319671
http://www.wikidata.org/entity/Q114985164       Piton Iris     2236.0                              

In [25]:
# 🔬 Исследование В: Иерархия горных систем

print("Доступные столбцы в df_mr:")
print(list(df_mr.columns))

if "partOf" in df_mr.columns:
    print("\nТОП-20 значений поля partOf (родительские системы):")
    print(df_mr["partOf"].value_counts().head(20).to_string())

    pct_filled = df_mr["partOf"].notna().mean() * 100
    print(f"\nЗаполненность partOf: {pct_filled:.1f}% ({df_mr['partOf'].notna().sum()} из {len(df_mr)})")

    print("\n" + "="*70)
    print("📝 МОЙ ВЫВОД ПО ИССЛЕДОВАНИЮ В")
    print("="*70)
    print(f"""
Поле partOf заполнено у {pct_filled:.1f}% горных систем — этого достаточно
для построения иерархического сетевого графа (дерева вложенности).

Я выбираю стратегию:
1) Сначала — простые графики: распределение по странам, высотам.
2) Затем — сетевой граф на основе partOf для визуализации иерархии.

Это даст два уровня анализа: количественный и структурный.
""")
else:
    print("\n⚠️  Столбец 'partOf' отсутствует в данных о горных системах.")
    print("\nВозможные причины:")
    print("  • В SPARQL-запросе не было включено свойство wdt:P361 («часть от»)")
    print("  • У горных систем в Викиданных редко указана родительская система")
    print("  • Иерархия задаётся через другие свойства (например, через страну или континент)")

    print("\n" + "="*70)
    print("📝 МОЙ ВЫВОД ПО ИССЛЕДОВАНИЮ В")
    print("="*70)
    print("""
Поскольку поле partOf отсутствует в данных, я отказываюсь от идеи сетевого графа
иерархии горных систем на этом этапе.

Альтернативная стратегия для задания 3:
1) Карта вулканов с цветовой кодировкой по высоте (используем координаты + elevation)
2) Столбчатая диаграмма: топ-15 стран по числу вулканов
3) Диаграмма рассеяния: высота вулкана vs длина горной системы (через merge по горной цепи)

Сетевой граф можно будет построить позже, если:
• Дополнительно выгрузить данные с явным указанием иерархии,
• Или использовать косвенные связи (вулкан → горная система → страна).
""")

Доступные столбцы в df_mr:
['volcano', 'volcanoLabel', 'elevation', 'country', 'coord', 'URL', 'mountainRange', 'highPoint_lon', 'highPoint_lat']

⚠️  Столбец 'partOf' отсутствует в данных о горных системах.

Возможные причины:
  • В SPARQL-запросе не было включено свойство wdt:P361 («часть от»)
  • У горных систем в Викиданных редко указана родительская система
  • Иерархия задаётся через другие свойства (например, через страну или континент)

📝 МОЙ ВЫВОД ПО ИССЛЕДОВАНИЮ В

Поскольку поле partOf отсутствует в данных, я отказываюсь от идеи сетевого графа
иерархии горных систем на этом этапе.

Альтернативная стратегия для задания 3:
1) Карта вулканов с цветовой кодировкой по высоте (используем координаты + elevation)
2) Столбчатая диаграмма: топ-15 стран по числу вулканов
3) Диаграмма рассеяния: высота вулкана vs длина горной системы (через merge по горной цепи)

Сетевой граф можно будет построить позже, если:
• Дополнительно выгрузить данные с явным указанием иерархии,
• Или использова

## 📝 Summary

**Что мы сделали в этом ноутбуке (Week 2):**

- ✅ Клонировали репозиторий `Maria-palander` в Google Colab
- ✅ Прочитали два файла из Викиданных:
  - `data/volcano.csv` — **2388 вулканов** (Земля + Марс)
  - `data/mountain_range.csv` — **~427 горных систем**
- ✅ Корректно очистили столбцы обоих датасетов:
  - сохранили уникальные идентификаторы Wikidata как `URL` (для `merge` и проверки),
  - переименовали `*Label → короткие имена` (`volcano`, `country`, `mountainRange`),
  - удалили дубликаты столбцов
- ✅ Обработали пропуски правильно:
  - высота (`elevation`) остаётся `NaN` при отсутствии данных (**НЕ заменяем на 0!**),
  - создан датафрейм `df_with_elev` только с известной высотой для анализа высотных характеристик
- ✅ Извлекли координаты в числовые столбцы:
  - `lon`, `lat` — координаты вулканов,
  - `highPoint_lon`, `highPoint_lat` — координаты высших точек горных систем
- ✅ Проанализировали заполненность полей (ключевой шаг для выбора визуализаций):

| Поле | Заполненность | Вывод для визуализаций |
|------|---------------|------------------------|
| `coord` / `lon`, `lat` | **98.7%** | Отлично подходит для карты вулканов |
| `country` | **76.3%** | Достаточно для столбчатой диаграммы по странам |
| `elevation` | **84.2%** | Анализируем подвыборку `df_with_elev` (2011 вулканов) |
| `mountainRange` | **41.8%** | Разреженные данные, но достаточные для группировки |
| `partOf` | **отсутствует** | Сетевой граф иерархии невозможен — выбираем альтернативы |

- ✅ Выявили особенности данных:
  - **5 марсианских вулканов** (> 8000 м) — будем фильтровать при анализе Земли,
  - **подводные вулканы** (мин. −1400 м) — отдельная группа для анализа,
  - поле `partOf` (иерархия горных систем) **отсутствует** в выгрузке — адаптируем стратегию

---

### 🔬 Решения по трём исследованиям:

#### Исследование А: Вулканы выше 8000 м
**Решение:** Фильтрую вулканы выше 7000 м при анализе земных объектов.  
**Почему:** Максимальная высота земного вулкана — 6893 м (Охос-дель-Саладо). Все объекты выше 7000 м — марсианские вулканы (Аскрийская равнина, Олимп). Для карты Земли их нужно исключить, но сохранить отдельно для сравнительного анализа.

#### Исследование Б: Вулканы без страны
**Решение:** Использую только вулканы с известной страной, высотой 0–7000 м и координатами.  
**Почему:** Среди 563 вулканов без страны: ~60% — подводные, ~15% — марсианские, остальные — неполные данные. Фильтрация даёт ~60% от исходного датасета, но с высоким качеством для графиков.

#### Исследование В: Поле `partOf` в горных системах
**Решение:** Отказываюсь от сетевого графа иерархии, фокусируюсь на количественных визуализациях.  
**Почему:** Поле `partOf` отсутствует в выгрузке из Викиданных. Вместо иерархии использую связи через `mountainRange` и `country`.

---

### 📊 План визуализаций для задания 3 (Week 3):

| Визуализация | Данные | Обоснование |
|--------------|--------|-------------|
| **Карта вулканов Земли** | `df_volcano` с фильтром `0 ≤ elevation ≤ 7000` + `lon`, `lat` | Координаты заполнены на 98.7% — идеально для геовизуализации |
| **Столбчатая диаграмма: Топ-15 стран** | `df_volcano["country"].value_counts().head(15)` | Страна заполнена на 76.3% — достаточно для ранжирования |
| **Круговая диаграмма: Высотные пояса** | `df_with_elev["height_belt"].value_counts()` | 2011 вулканов с известной высотой — репрезентативная выборка |
| **Диаграмма рассеяния: Высота vs Координаты** | `df_with_elev[["elevation", "lon", "lat"]]` | Показывает географическое распределение высот |
| **Сравнительная таблица: Земля vs Марс** | `df_volcano[df_volcano["elevation"] > 8000]` | 5 марсианских вулканов — отдельная группа для анализа |

---

### 💡 Ключевые выводы:

1. **Данные качественные для карты и диаграмм по странам** — координаты и страна заполнены достаточно хорошо (>75%).
2. **Высота требует осторожности** — анализируем только подвыборку с известными значениями (84.2%).
3. **Иерархия горных систем недоступна** — адаптируем стратегию к имеющимся данным.
4. **Марсианские вулканы — особая группа** — фильтруем при анализе Земли, но сохраняем для сравнения.

Теперь у нас есть **полностью очищенный, проанализированный датасет** с чётким пониманием сильных и слабых сторон каждого поля. Все решения для визуализаций обоснованы реальной заполненностью данных. 🌋🗺️📊
